<a href="https://colab.research.google.com/github/codematser69/candidate-ranking--dashboard/blob/main/panda_tech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CELL 1: Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# Confirm your folder name matches EXACTLY (case + hyphen/underscore sensitive)
# CONFIRMED real folder name: redrob_hackathon (underscore, not hyphen)
import os
print(os.listdir('/content/drive/MyDrive/redrob_hackathon'))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['candidates.jsonl', 'job_description.docx', 'sample_candidates.json', 'validate_submission.py', 'submission_spec.docx', 'submission_metadata_template.yaml', 'candidate_schema.json', 'redrob_signals_doc.docx', 'sample_submission.csv', 'output']


In [ ]:
# ============================================================
# CELL 2: Install only what's missing on Colab
# ============================================================
!pip install -q lightgbm sentence-transformers
print("Libraries installed")


Libraries installed


In [ ]:
# ============================================================
# CELL 3: Imports & Config
# ============================================================
import json
import os
import re
import math
import time
import warnings
from datetime import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings("ignore")

# PATHS — CONFIRMED real folder name: redrob_hackathon (underscore, not hyphen)
DRIVE_BASE = "/content/drive/MyDrive/redrob_hackathon"
CANDIDATES_PATH = f"{DRIVE_BASE}/candidates.jsonl"     # NOTE: .jsonl, not .jsonl.gz
OUTPUT_DIR = f"{DRIVE_BASE}/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Config ready")
print(f"Dataset path: {CANDIDATES_PATH}")
print(f"Output path : {OUTPUT_DIR}")
print(f"File exists : {os.path.exists(CANDIDATES_PATH)}")   # MUST print True before proceeding to Cell 4

Config ready
Dataset path: /content/drive/MyDrive/redrob_hackathon/candidates.jsonl
Output path : /content/drive/MyDrive/redrob_hackathon/output
File exists : True


In [ ]:
# ============================================================
# CELL 4: Load all 100K candidates (plain .jsonl, no gzip needed)
# ============================================================
def load_candidates(path):
    candidates = []
    with open(path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="Loading candidates", unit=" lines"):
            line = line.strip()
            if line:
                candidates.append(json.loads(line))
    return candidates

start = time.time()
candidates = load_candidates(CANDIDATES_PATH)
print(f"Loaded {len(candidates):,} candidates in {time.time()-start:.1f}s")


Loading candidates: 100000 lines [00:16, 6076.83 lines/s]

Loaded 100,000 candidates in 16.5s


In [ ]:
# ============================================================
# CELL 5: Explore one candidate record — confirm structure matches this guide
# ============================================================
sample = candidates[0]
print(json.dumps(sample, indent=2, default=str)[:2000])


{
  "candidate_id": "CAND_0000001",
  "profile": {
    "anonymized_name": "Ira Vora",
    "headline": "Backend Engineer | SQL, Spark, Cloud",
    "summary": "Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid \u2014 Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side \u2014 Python, SQL, Spark, Airflow, warehouse design \u2014 and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.",
    "location": "Toronto",
    "country": "Canada",
    "years_of_experience": 6.9,
    "current_title": "Backend Engineer",
    "current_company": "Mindtree",
    "current_company_size": "1

In [ ]:
# CELL 6: Confirm top-level keys match what this guide expects
all_keys = set()
for c in candidates[:1000]:
    all_keys.update(c.keys())
print("Top-level keys found:", sorted(all_keys))
# Expected: candidate_id, profile, career_history, education,
#           skills, certifications, languages, redrob_signals

Top-level keys found: ['candidate_id', 'career_history', 'certifications', 'education', 'languages', 'profile', 'redrob_signals', 'skills']


In [ ]:
# CELL 7: Confirm redrob_signals keys (should be 23 signals)
print(json.dumps(candidates[0]["redrob_signals"], indent=2))

{
  "profile_completeness_score": 86.9,
  "signup_date": "2025-10-16",
  "last_active_date": "2026-05-20",
  "open_to_work_flag": true,
  "profile_views_received_30d": 23,
  "applications_submitted_30d": 2,
  "recruiter_response_rate": 0.34,
  "avg_response_time_hours": 177.8,
  "skill_assessment_scores": {
    "NLP": 38.8,
    "Image Classification": 64.8,
    "Fine-tuning LLMs": 41.6,
    "Speech Recognition": 53.7
  },
  "connection_count": 356,
  "endorsements_received": 35,
  "notice_period_days": 60,
  "expected_salary_range_inr_lpa": {
    "min": 18.7,
    "max": 36.1
  },
  "preferred_work_mode": "onsite",
  "willing_to_relocate": false,
  "github_activity_score": 9.2,
  "search_appearance_30d": 249,
  "saved_by_recruiters_30d": 4,
  "interview_completion_rate": 0.71,
  "offer_acceptance_rate": 0.58,
  "verified_email": true,
  "verified_phone": true,
  "linkedin_connected": false
}


In [ ]:
# ============================================================
# CELL 8: Flatten all candidates into a Pandas DataFrame
# Every field path below matches the REAL nested schema.
# ============================================================

def days_since(date_str, reference_date="2026-06-20"):
    """Compute days between a date string and the reference date (today)."""
    if not date_str:
        return 999
    try:
        d = datetime.strptime(date_str, "%Y-%m-%d")
        ref = datetime.strptime(reference_date, "%Y-%m-%d")
        return (ref - d).days
    except Exception:
        return 999

def get_best_education_tier(education_list):
    """Education tier lives PER DEGREE. Take the best (lowest number) tier found."""
    if not education_list:
        return 3  # default mid-tier if missing
    tiers = []
    for edu in education_list:
        t = edu.get("tier", "")
        match = re.search(r"(\d+)", str(t))
        if match:
            tiers.append(int(match.group(1)))
    return min(tiers) if tiers else 3

rows = []
for c in tqdm(candidates, desc="Flattening to DataFrame"):
    cid = c.get("candidate_id", "")
    profile = c.get("profile", {}) or {}
    signals = c.get("redrob_signals", {}) or {}
    career_history = c.get("career_history", []) or []
    education = c.get("education", []) or []
    skills = c.get("skills", []) or []

    row = {
        "candidate_id"        : cid,
        "current_title"       : profile.get("current_title", ""),
        "current_company"     : profile.get("current_company", ""),
        "current_industry"    : profile.get("current_industry", ""),
        "current_company_size": profile.get("current_company_size", ""),
        "location_city"       : profile.get("location", ""),
        "country"             : profile.get("country", ""),
        "total_exp_years"     : profile.get("years_of_experience", 0) or 0,
        "headline"            : profile.get("headline", ""),
        "summary"             : profile.get("summary", ""),

        # From redrob_signals, NOT top-level
        "notice_period_days"  : signals.get("notice_period_days", 999) or 999,
        "willing_to_relocate" : int(bool(signals.get("willing_to_relocate", False))),
        "open_to_work_flag"   : int(bool(signals.get("open_to_work_flag", False))),
        "last_active_days_ago": days_since(signals.get("last_active_date", "")),
        "recruiter_response_rate": signals.get("recruiter_response_rate", 0) or 0,
        "github_activity_score": signals.get("github_activity_score", 0) or 0,
        "profile_completeness_score": signals.get("profile_completeness_score", 0) or 0,
        "interview_completion_rate": signals.get("interview_completion_rate", 0) or 0,
        "offer_acceptance_rate": signals.get("offer_acceptance_rate", 0) or 0,

        # Education tier extracted from nested list
        "education_tier"      : get_best_education_tier(education),

        # Raw blobs kept for later inspection / reasoning generation
        "skills_raw"          : json.dumps(skills, default=str),
        "career_history_raw"  : json.dumps(career_history, default=str),
        "education_raw"       : json.dumps(education, default=str),
        "full_text"           : "",
    }

    # Pull EVERY redrob_signal into its own column (sig_ prefix), skipping nested dicts/dates
    for sig_key, sig_val in signals.items():
        if isinstance(sig_val, (int, float, bool)):
            row[f"sig_{sig_key}"] = float(sig_val)
        # skip dicts (skill_assessment_scores, expected_salary_range_inr_lpa) and date strings here —
        # handle those separately if you want them as features later

    # Build searchable full-text blob for TF-IDF / keyword matching
    skills_list = [s.get("name", "") for s in skills if isinstance(s, dict)]

    career_text = []
    for job in career_history:
        if isinstance(job, dict):
            career_text.append(job.get("title", ""))
            career_text.append(job.get("description", ""))

    education_text = []
    for edu in education:
        if isinstance(edu, dict):
            education_text.append(edu.get("degree", ""))
            education_text.append(edu.get("field_of_study", ""))
            education_text.append(edu.get("institution", ""))

    row["full_text"] = " ".join(filter(None, [
        row["current_title"],
        row["headline"],
        " ".join(skills_list),
        " ".join(career_text),
        " ".join(education_text),
        row["summary"],
    ])).lower()

    rows.append(row)

df = pd.DataFrame(rows)
print(f"DataFrame created: {df.shape}")

# Sanity checks
print(f"\nTotal rows         : {len(df):,}")
print(f"Unique IDs          : {df['candidate_id'].nunique():,}")
print(f"Experience range    : {df['total_exp_years'].min():.1f} - {df['total_exp_years'].max():.1f}")
print(f"Notice period range : {df['notice_period_days'].min()} - {df['notice_period_days'].max()}")
sig_cols = [c for c in df.columns if c.startswith("sig_")]
print(f"Behavioral signal columns: {len(sig_cols)}")

# Save checkpoint to Drive so you don't have to re-flatten 100K rows every session
df.to_parquet(f"{OUTPUT_DIR}/candidates_flat.parquet", index=False)
print(f"\nCheckpoint saved: candidates_flat.parquet")


Flattening to DataFrame: 100%|██████████| 100000/100000 [00:21<00:00, 4722.60it/s]


DataFrame created: (100000, 42)

Total rows         : 100,000
Unique IDs          : 100,000
Experience range    : 1.0 - 16.9
Notice period range : 15 - 999
Behavioral signal columns: 18


In [ ]:
df = pd.read_parquet(f"{OUTPUT_DIR}/candidates_flat.parquet")

chunk-2: Feature Engineering & Signal Extraction